# Enhanced Linear Regression with Visualizations and Pipeline

This notebook improves the simple model by including:
- Numeric and categorical features
- OneHotEncoding
- StandardScaler
- Pipeline
- Visual evaluation
- Model saving

## 1. Import required libraries

In [ ]:
# pandas is used for data loading and manipulation
import pandas as pd

# matplotlib is used for simple visualizations
import matplotlib.pyplot as plt

# pickle is used to save the trained model
import pickle

# train_test_split splits data into training and testing sets
from sklearn.model_selection import train_test_split

# LinearRegression is the regression algorithm
from sklearn.linear_model import LinearRegression

# Evaluation metrics for regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ColumnTransformer applies different preprocessing to different column types
from sklearn.compose import ColumnTransformer

# OneHotEncoder converts categorical values into numeric columns
# StandardScaler scales numeric columns
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Pipeline combines preprocessing and model training in one flow
from sklearn.pipeline import Pipeline

## 2. Load dataset

In [ ]:
# Load dataset from CSV file
df = pd.read_csv('employee.csv')

# Display first five records
df.head()

## 3. Understand the dataset

In [ ]:
# Check dataset size
print('Rows and columns:', df.shape)

# Check column names and data types
df.info()

In [ ]:
# Check missing values before cleaning
print(df.isnull().sum())

## 4. Basic cleaning

In [ ]:
# Remove leading/trailing spaces from column names
df.columns = df.columns.str.strip()

# Remove leading/trailing spaces from object/string columns
for column in df.select_dtypes(include='object').columns:
    df[column] = df[column].str.strip()

# Replace '?' with proper missing values
df = df.replace('?', pd.NA)

# Drop rows with missing values for simplicity
df = df.dropna()

print('Cleaned dataset shape:', df.shape)

## 5. Simple visualizations

In [ ]:
# Histogram shows distribution of employee working hours
plt.figure(figsize=(8, 5))
plt.hist(df['hours-per-week'], bins=20)
plt.title('Distribution of Hours Per Week')
plt.xlabel('Hours Per Week')
plt.ylabel('Number of Employees')
plt.show()

In [ ]:
# Bar chart shows count of employees by education level
education_counts = df['education'].value_counts().head(10)

plt.figure(figsize=(10, 5))
plt.bar(education_counts.index, education_counts.values)
plt.title('Top 10 Education Categories')
plt.xlabel('Education')
plt.ylabel('Employee Count')
plt.xticks(rotation=45)
plt.show()

## 6. Target and feature selection

In [ ]:
# Target column to predict
target = 'hours-per-week'

# Remove target column from input features
X = df.drop(columns=[target])

# Store target column separately
y = df[target]

# Identify numeric columns automatically
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Identify categorical columns automatically
categorical_features = X.select_dtypes(include='object').columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

## 7. Train-test split

In [ ]:
# Split data into train and test sets
# Training data is used to learn patterns
# Testing data is used to check model performance on unseen data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

## 8. Build preprocessing pipeline

In [ ]:
# Numeric transformer scales numeric columns
numeric_transformer = StandardScaler()

# Categorical transformer converts categories to numeric dummy variables
# handle_unknown='ignore' prevents errors when new categories appear in test data
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# ColumnTransformer applies numeric preprocessing to numeric columns
# and categorical preprocessing to categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## 9. Create Linear Regression pipeline

In [ ]:
# Pipeline first preprocesses data, then trains Linear Regression model
model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ]
)

## 10. Train the model

In [ ]:
# Train the complete pipeline using training data
model.fit(X_train, y_train)

print('Model training completed')

## 11. Predict and evaluate

In [ ]:
# Predict hours-per-week for test data
y_pred = model.predict(X_test)

# Calculate regression metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print('MAE:', mae)
print('MSE:', mse)
print('RMSE:', rmse)
print('R2 Score:', r2)

## 12. Actual vs predicted visualization

In [ ]:
# Compare actual and predicted values visually
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.4)
plt.xlabel('Actual Hours Per Week')
plt.ylabel('Predicted Hours Per Week')
plt.title('Actual vs Predicted Hours Per Week')
plt.show()

## 13. Residual plot

In [ ]:
# Residual means actual value minus predicted value
residuals = y_test - y_pred

plt.figure(figsize=(7, 5))
plt.scatter(y_pred, residuals, alpha=0.4)
plt.axhline(y=0, linestyle='--')
plt.xlabel('Predicted Hours Per Week')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.show()

## 14. Save the trained model

In [ ]:
# Save the complete trained pipeline as a pickle file
# This includes preprocessing and Linear Regression model
with open('linear_regression_employee_model.pkl', 'wb') as file:
    pickle.dump(model, file)

print('Model saved successfully')